# 02. Обучение нейросетей в Google Colab (GPU)

**Перед запуском:** меню *Среда выполнения → Сменить среду выполнения → T4 GPU*.

В Google Drive должна быть папка `skin-lesion/` с файлами `pad_ufes_256.zip` и `metadata_split.csv`.
Все результаты сохраняются **в Google Drive** (`skin-lesion/outputs`), поэтому при обрыве сессии ничего не теряется.
Логика обучения лежит в `src/train_image.py` в репозитории, а этот ноутбук только запускает её на GPU.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/skin-lesion"
OUT = f"{DRIVE}/outputs"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone https://github.com/Zhar04/skin-lesion-triage.git /content/project
%cd /content/project
!pip install -q timm grad-cam

Данные копируем на локальный диск Colab: чтение с него намного быстрее, чем с Drive.

In [ ]:
!mkdir -p data/processed
!unzip -q -o "{DRIVE}/pad_ufes_256.zip" -d data/processed
!cp "{DRIVE}/metadata_split.csv" data/processed/
!ls data/processed/images_256 | wc -l   # ожидаем 2298

### Быстрая проверка (1 минута): всё запускается?

In [ ]:
!python src/train_image.py --max-images 150 --epochs 1 --out /content/smoke_test

### 4.2 Замороженная CNN (только голова учится): быстрый baseline по фото
*Зачем:* показывает, сколько даёт само по себе представление ImageNet без дообучения.

In [ ]:
for seed in [42, 43, 44]:
    !python src/train_image.py --freeze --epochs 15 --seed {seed} --out "{OUT}"

### 4.3 Fine-tuning EfficientNet-B0: основная модель по фото
Смотрите на вывод: если `train AUC` растёт, а `val AUC` падает, это переобучение (early stopping остановит).

In [ ]:
for seed in [42, 43, 44]:
    !python src/train_image.py --seed {seed} --out "{OUT}"

### 5.2 Feature-level fusion: фото + клиника (главная модель проекта)

In [ ]:
for seed in [42, 43, 44]:
    !python src/train_image.py --clinical safe --seed {seed} --out "{OUT}"

### 4.5 (Если есть время) Другая архитектура: ResNet50
*Зачем:* ещё одна строка в таблице сравнения, проверка, что вывод не зависит от одной архитектуры.

In [ ]:
for seed in [42, 43, 44]:
    !python src/train_image.py --arch resnet50 --seed {seed} --out "{OUT}"

### Готово
Скачайте папку `skin-lesion/outputs` из Google Drive и **объедините** её с локальной `outputs/` в проекте
(там уже лежат предсказания клинических моделей). Дальше по ROADMAP: этапы 5.1 и 6 выполняются на ноутбуке.

In [ ]:
!ls "{OUT}/preds" "{OUT}/models"